### Hodgkin-Huxley Neuron Voltage Clamp Simulator

This notebook provides an interactive simulation of a Hodgkin-Huxley neuron under voltage clamp conditions. You can adjust the clamped voltage using a slider and observe the resulting currents.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import odeint
import ipywidgets as widgets
from IPython.display import display

# Define Hodgkin-Huxley parameters
C_m = 1.0   # Membrane capacitance (uF/cm^2)
g_Na = 120.0 # Maximum sodium conductance (mS/cm^2)
g_K = 36.0  # Maximum potassium conductance (mS/cm^2)
g_L = 0.3   # Maximum leak conductance (mS/cm^2)
E_Na = 50.0 # Sodium reversal potential (mV)
E_K = -77.0 # Potassium reversal potential (mV)
E_L = -54.387 # Leak reversal potential (mV)

print("Hodgkin-Huxley parameters initialized.")

Hodgkin-Huxley parameters initialized.


In [ ]:
# Gating variable functions (alpha and beta rates)
def alpha_n(V):
    return 0.01 * (V + 55) / (1 - np.exp(-(V + 55) / 10))

def beta_n(V):
    return 0.125 * np.exp(-(V + 65) / 80)

def alpha_m(V):
    return 0.1 * (V + 40) / (1 - np.exp(-(V + 40) / 10))

def beta_m(V):
    return 4.0 * np.exp(-(V + 65) / 18)

def alpha_h(V):
    return 0.07 * np.exp(-(V + 65) / 20)

def beta_h(V):
    return 1.0 / (1 + np.exp(-(V + 35) / 10))

print("Gating variable functions defined.")

Gating variable functions defined.


In [ ]:
# Hodgkin-Huxley differential equations for voltage clamp
def hh_vc_ode(y, t, V_clamped):
    n, m, h = y

    # Calculate derivatives of gating variables
    dn_dt = alpha_n(V_clamped) * (1 - n) - beta_n(V_clamped) * n
    dm_dt = alpha_m(V_clamped) * (1 - m) - beta_m(V_clamped) * m
    dh_dt = alpha_h(V_clamped) * (1 - h) - beta_h(V_clamped) * h

    return [dn_dt, dm_dt, dh_dt]

print("Hodgkin-Huxley ODE for voltage clamp defined.")

Hodgkin-Huxley ODE for voltage clamp defined.


In [ ]:
# Function to run the simulation and plot results
def run_hh_voltage_clamp_simulation(clamped_voltage):
    # Time parameters
    dt = 0.01  # Time step (ms)
    t_start = 0
    t_clamp_start = 10  # ms
    t_clamp_duration = 25 # ms
    t_clamp_end = t_clamp_start + t_clamp_duration
    t_end = t_clamp_end + 15 # ms (return to baseline for 15ms)
    time = np.arange(t_start, t_end, dt)

    # Baseline voltage (resting potential)
    V_rest = -65.0 # mV

    # Initial conditions for gating variables at V_rest
    n0 = alpha_n(V_rest) / (alpha_n(V_rest) + beta_n(V_rest))
    m0 = alpha_m(V_rest) / (alpha_m(V_rest) + beta_m(V_rest))
    h0 = alpha_h(V_rest) / (alpha_h(V_rest) + beta_h(V_rest))
    y0 = [n0, m0, h0]

    # Arrays to store results
    V_arr = np.zeros_like(time)
    I_Na_arr = np.zeros_like(time)
    I_K_arr = np.zeros_like(time)
    I_L_arr = np.zeros_like(time)
    I_ion_arr = np.zeros_like(time)
    n_arr = np.zeros_like(time)
    m_arr = np.zeros_like(time)
    h_arr = np.zeros_like(time)

    # Simulate in segments due to voltage clamp step
    # Segment 1: Baseline voltage
    idx_baseline = np.where(time < t_clamp_start)[0]
    if len(idx_baseline) > 0:
        t_baseline = time[idx_baseline]
        sol_baseline = odeint(hh_vc_ode, y0, t_baseline, args=(V_rest,))
        n_arr[idx_baseline], m_arr[idx_baseline], h_arr[idx_baseline] = sol_baseline.T
        V_arr[idx_baseline] = V_rest
        y0_clamped = sol_baseline[-1, :]
    else:
        y0_clamped = y0

    # Segment 2: Clamped voltage
    idx_clamped = np.where((time >= t_clamp_start) & (time < t_clamp_end))[0]
    if len(idx_clamped) > 0:
        t_clamped = time[idx_clamped]
        sol_clamped = odeint(hh_vc_ode, y0_clamped, t_clamped, args=(clamped_voltage,))
        n_arr[idx_clamped], m_arr[idx_clamped], h_arr[idx_clamped] = sol_clamped.T
        V_arr[idx_clamped] = clamped_voltage
        y0_return = sol_clamped[-1, :]
    else:
        y0_return = y0_clamped

    # Segment 3: Return to baseline voltage
    idx_return = np.where(time >= t_clamp_end)[0]
    if len(idx_return) > 0:
        t_return = time[idx_return]
        sol_return = odeint(hh_vc_ode, y0_return, t_return, args=(V_rest,))
        n_arr[idx_return], m_arr[idx_return], h_arr[idx_return] = sol_return.T
        V_arr[idx_return] = V_rest

    # Calculate currents for the entire simulation
    for i in range(len(time)):
        V = V_arr[i]
        n = n_arr[i]
        m = m_arr[i]
        h = h_arr[i]

        I_Na = g_Na * m**3 * h * (V - E_Na)
        I_K = g_K * n**4 * (V - E_K)
        I_L = g_L * (V - E_L)
        I_ion = I_Na + I_K + I_L

        I_Na_arr[i] = I_Na
        I_K_arr[i] = I_K
        I_L_arr[i] = I_L
        I_ion_arr[i] = I_ion

    # Plotting
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True)

    # Voltage plot
    ax1.plot(time, V_arr, 'b')
    ax1.set_ylabel('Voltage (mV)')
    ax1.set_title('Hodgkin-Huxley Voltage Clamp Simulation')
    ax1.grid(True)
    ax1.axvline(x=t_clamp_start, color='gray', linestyle='--', label='Clamp Start')
    ax1.axvline(x=t_clamp_end, color='gray', linestyle='--', label='Clamp End')
    ax1.legend()

    # Current plot
    ax2.plot(time, I_Na_arr, 'r', label='I_Na')
    ax2.plot(time, I_K_arr, 'g', label='I_K')
    ax2.plot(time, I_L_arr, 'k', label='I_L')
    ax2.plot(time, I_ion_arr, 'm', linestyle='--', label='I_ion (Total)')
    ax2.set_xlabel('Time (ms)')
    ax2.set_ylabel('Current (uA/cm^2)') # Note: In HH, current is typically per unit area, not nA directly
    ax2.set_ylim(-1000, 1000) # Adjust limits as needed
    ax2.grid(True)
    ax2.legend()

    plt.tight_layout()
    plt.show()

print("Simulation function defined.")

Simulation function defined.


In [ ]:
# Create interactive slider for voltage clamp
v_clamp_slider = widgets.FloatSlider(
    value=-65,  # Starting voltage
    min=-90,    # Minimum voltage as per request
    max=50,     # Maximum voltage as per request
    step=1,     # Step size
    description='Clamped Voltage (mV):',
    continuous_update=False,
    orientation='horizontal',
    readout=True,
    readout_format='.1f',
)

# Link the slider to the simulation function
interactive_plot = widgets.interactive(run_hh_voltage_clamp_simulation, clamped_voltage=v_clamp_slider)

# Display the widget and plot
display(interactive_plot)

print("Interactive simulator created.")

interactive(children=(FloatSlider(value=-65.0, continuous_update=False, description='Clamped Voltage (mV):', m…

Interactive simulator created.
